In [2]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
data = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/IMDB_dataset.csv")

In [5]:
import torch
torch.cuda.is_available()

True

In [ ]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
data.shape

(50000, 2)

In [ ]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


## One Hot Encoding

### Label Encoder

In [ ]:
data.replace({"sentiment": {"positive":1, "negative":0}}, inplace=True)

In [ ]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


### Data Preprocessing

In [6]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [10]:
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

In [11]:
train_data.shape

(40000, 2)

In [12]:
test_data.shape

(10000, 2)

In [13]:
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data['review'])


In [ ]:
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data['review']), maxlen=500)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data['review']), maxlen=500)

In [ ]:
X_train

array([[   0,    0,    0, ...,  205,  351, 3856],
       [   0,    0,    0, ...,   89,  103,    9],
       [   0,    0,    0, ...,    2,  710,   62],
       ...,
       [   0,    0,    0, ..., 1641,    2,  603],
       [   0,    0,    0, ...,  245,  103,  125],
       [   0,    0,    0, ...,   70,   73, 2062]], dtype=int32)

In [ ]:
X_test

array([[   0,    0,    0, ...,  995,  719,  155],
       [   0,    0,    0, ...,  380,    7,    7],
       [   0,    0,    0, ...,   50, 1088,   96],
       ...,
       [   0,    0,    0, ...,  125,  200, 3241],
       [   0,    0,    0, ..., 1066,    1, 2305],
       [   0,    0,    0, ...,    1,  332,   27]], dtype=int32)

In [ ]:
Y_train = train_data['sentiment']
Y_test = test_data['sentiment']

### Model Building

In [ ]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=128, input_length=500))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation='sigmoid'))

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.fit(X_train, Y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 488s 972ms/step - accuracy: 0.8206 - loss: 0.4111 - val_accuracy: 0.8626 - val_loss: 0.3261
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 507s 982ms/step - accuracy: 0.8703 - loss: 0.3278 - val_accuracy: 0.8534 - val_loss: 0.3506
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 497s 993ms/step - accuracy: 0.8859 - loss: 0.2896 - val_accuracy: 0.8605 - val_loss: 0.3310
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 506s 1s/step - accuracy: 0.8761 - loss: 0.3034 - val_accuracy: 0.7785 - val_loss: 0.4581
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 491s 983ms/step - accuracy: 0.8591 - loss: 0.3308 - val_accuracy: 0.8775 - val_loss: 0.3186


In [ ]:
loss, accuracy = model.evaluate(X_test, Y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 90s 286ms/step - accuracy: 0.8823 - loss: 0.3019


In [ ]:
print(loss)

0.30383044481277466


In [ ]:
print(accuracy)

0.8840000033378601


In [ ]:
### Building predictive System

In [ ]:
def predictive_system(review):
  seq = tokenizer.texts_to_sequences([review])
  padded = pad_sequences(seq, maxlen=200)
  pred = model.predict(padded)
  sentiment = "Positive" if pred > 0.5 else "Negative"
  return sentiment

In [ ]:
predictive_system("this movie is fantastic and amazing")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step


'Positive'

In [ ]:
predictive_system("a threlling and advanture with stunning visual")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step


'Positive'

In [ ]:
predictive_system("a visual masterpiece")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 130ms/step


'Positive'

In [ ]:
predictive_system("overall long and slow")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 141ms/step


'Positive'

In [ ]:
## Saving Model

In [ ]:
model.save("model.h5")

In [14]:
import joblib
joblib.dump(tokenizer, '/content/drive/MyDrive/tokenizer.pkl')


['/content/drive/MyDrive/tokenizer.pkl']